<a href="https://colab.research.google.com/github/gloriacitizen00-dev/Docs/blob/main/MysticalWeatherCalendar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
import pandas as pd
from datetime import datetime
from IPython.display import display, HTML

# --- Step 1: Get Alaska Weather Data (Open-Meteo) ---
lat = 61.2176   # Anchorage latitude
lon = -149.8997 # Anchorage longitude

URL = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&daily=temperature_2m_max,temperature_2m_min,weathercode&timezone=America/Anchorage&forecast_days=16"

response = requests.get(URL)
data = response.json()

# --- Step 2: Organize Daily Forecast ---
dates = data["daily"]["time"]
temps_max = data["daily"]["temperature_2m_max"]
temps_min = data["daily"]["temperature_2m_min"]
conditions = data["daily"]["weathercode"]

weather_map = {
    0: "Clear sky", 1: "Mainly clear", 2: "Partly cloudy", 3: "Overcast",
    45: "Fog", 48: "Rime fog",
    51: "Light drizzle", 61: "Light rain", 71: "Light snow",
    80: "Rain showers", 85: "Snow showers"
}

icon_map = {
    "Clear sky": "☀️", "Mainly clear": "🌤️", "Partly cloudy": "⛅", "Overcast": "☁️",
    "Fog": "🌫️", "Rime fog": "🌫️",
    "Light drizzle": "🌦️", "Light rain": "🌧️", "Light snow": "❄️",
    "Rain showers": "🌦️", "Snow showers": "🌨️", "Unknown": "❓"
}

color_map = {
    "Clear sky": "#FFD700",   # golden
    "Mainly clear": "#FFE87C",
    "Partly cloudy": "#E0E0E0",
    "Overcast": "#A9A9A9",    # gray
    "Fog": "#D3D3D3",
    "Rime fog": "#D3D3D3",
    "Light drizzle": "#87CEFA",
    "Light rain": "#87CEEB",
    "Light snow": "#ADD8E6",  # pale blue
    "Rain showers": "#87CEFA",
    "Snow showers": "#B0E0E6"
}

summary = []
for i in range(len(dates)):
    date = datetime.strptime(dates[i], "%Y-%m-%d").date()
    avg_temp = (temps_max[i] + temps_min[i]) / 2
    condition = weather_map.get(conditions[i], "Unknown")
    summary.append({"date": date, "avg_temp": avg_temp, "condition": condition})

df = pd.DataFrame(summary)

# --- Step 3: Build Responsive HTML Calendar ---
html = """
<style>
.calendar {
  display: grid;
  grid-template-columns: repeat(7, 1fr);
  gap: 10px;
  font-family: 'Segoe UI', sans-serif;
}
.day {
  padding: 10px;
  border-radius: 8px;
  text-align: center;
  color: #222;
}
.day h4 {
  margin: 5px 0;
}
.day p {
  margin: 2px 0;
}
</style>
<div class="calendar">
"""

for _, row in df.iterrows():
    bg = color_map.get(row["condition"], "white")
    icon = icon_map.get(row["condition"], "❓")
    html += f"""
    <div class="day" style="background:{bg}">
      <h4>{row['date'].strftime('%b %d')}</h4>
      <p>{row['avg_temp']:.1f}°C</p>
      <p style="font-size:24px">{icon}</p>
      <p>{row['condition']}</p>
    </div>
    """

html += "</div>"

display(HTML(html))
